# 03 · Avaliação TEST pareada (v4)

**Ambiente: GPU.** Só abre TEST depois dos 120 treinos SFT. Avalia cada adaptador no modo
`full`, com orçamento de 256 tokens: 26.400 gerações. Avaliações interrompidas
retomam do último item gravado; avaliações concluídas têm hash conferido e não são refeitas.


In [1]:
# Configuração operacional. Sementes, tamanhos, modelos e critérios estão fixos no protocolo v4.
STUDY_ID = "estudo-prospectivo-2026-v4"  # @param {type:"string"}
STORAGE_MODE = "auto"  # @param ["auto", "local"]
TECHNICAL_TEST = False  # @param {type:"boolean"}
MAX_JOBS_PER_SESSION = 12  # @param {type:"integer"}
CODE_COMMIT = "9066d811e656cb7c0d3c2120ba06c7ae356691da"
CODE_HASHES = {
 "__init__.py": "903b3bdb88f326693084e14633a5ebce1f862506124b58f67dc751c2a2c96f76",
 "avaliacao.py": "01b6f0f4f9e651dd2a1904eb19da2a8066a9366b976b768911722b9ada8d966f",
 "contrato.py": "fc9d9130d8edb89a56c7cf8b52dbe4ebda3b4d60d81f7964b769cb8515caa4b1",
 "estatistica.py": "7fd7615c0ead1e01c0d7e926e41c56a0b203b4ba493f3153362a9779c945bdfd",
 "modelos.py": "ee02866b8eac6db6928233805f50c985352d5c57253a3bfcf01d7c56fe910811",
 "monitor.py": "3635ed93e186e64dea87b0df428c8472434f40e2d243660b7ffb63ddce2a87f3",
 "runtime.py": "c1f4326e798b39e8b90fc6099d1c8da8b31e14a0ec9b46bf678c43c2a77fe3b0",
 "tarefas.py": "8b1a58ecc9df6b959741409f71811b7e90449bc41eedf7d860528f74732ba7fd",
 "treino.py": "a2d431c0ecaa64c618fad84609b43ca85377f48b57978d35961be212e73b8f84"
}
SUPPORT_HASHES = {
 "armazenamento.py": "66e04b1224550f8532311273ae78e118ffe1766654c6d0ee9c43ab2557d2482f",
 "sessao.py": "3c8c3774d66fce7fc0881bfb2db0e65d7bfbb2a2a38ef36f48373ddba29dde41"
}

import hashlib, importlib.util, os, re, subprocess, sys, time
from pathlib import Path
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,79}", STUDY_ID):
    raise ValueError("STUDY_ID aceita somente letras, números, _ e -.")
if TECHNICAL_TEST and not STUDY_ID.endswith("-teste-tecnico"):
    STUDY_ID += "-teste-tecnico"  # pasta separada: nunca se mistura ao estudo
try:
    from google.colab import drive
    ON_COLAB = True
except ImportError:
    drive, ON_COLAB = None, False
LOCAL = Path(os.environ.get("COT_LOCAL_ROOT", "/content/cot-v4" if ON_COLAB else Path.cwd() / "cot-v4"))
LOCAL.mkdir(parents=True, exist_ok=True)

def checkout(ref):
    """Obtém exatamente o commit fixado; não usa clones antigos nem a branch atual."""
    override = os.environ.get("COT_REPO_DIR")
    if override:
        return Path(override).resolve()
    repo = LOCAL / "source" / ref
    git = lambda *a, **k: subprocess.run(["git", "-C", str(repo), *a], capture_output=True, text=True, **k)
    if not (repo / ".git").is_dir():
        repo.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "init", "--quiet", str(repo)], check=True)
        git("remote", "add", "origin", "https://github.com/Kelvin-Marques-Cyber/cot-obfuscation-h2a.git")
    if git("cat-file", "-e", ref + "^{commit}").returncode:
        for attempt in range(1, 4):
            fetched = git("fetch", "--quiet", "--depth", "1", "origin", ref)
            if not fetched.returncode:
                break
            print(f"Falha de rede ao baixar o código (tentativa {attempt}/3): {fetched.stderr.strip()[-300:]}")
            time.sleep(5 * attempt)
        else:
            raise RuntimeError("Não foi possível baixar o commit " + ref + ". Verifique a conexão e execute de novo.")
    if git("checkout", "--quiet", "--detach", ref).returncode:
        raise RuntimeError("Checkout do commit fixado falhou: " + ref)
    if git("rev-parse", "HEAD").stdout.strip() != ref:
        raise RuntimeError("O código aberto não corresponde ao commit do notebook.")
    return repo

REPO = checkout(CODE_COMMIT)
PACKAGE = REPO / "experimento_cientifico_v4"
def load(name):
    path = PACKAGE / "colab" / name
    if hashlib.sha256(path.read_bytes()).hexdigest() != SUPPORT_HASHES[name]:
        raise RuntimeError("Suporte divergente do notebook: " + name + ". Reabra pelo link do GitHub.")
    # Nome único por commit: módulos de versões antigas na mesma sessão não são reutilizados.
    spec = importlib.util.spec_from_file_location(f"cotv4_{name[:-3]}_{CODE_COMMIT[:12]}", path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module
armazenamento, sessao = load("armazenamento.py"), load("sessao.py")
STATE = armazenamento.prepare_storage(LOCAL, STUDY_ID, on_colab=ON_COLAB, mode=STORAGE_MODE,
    storage_override=os.environ.get("COT_STORAGE_ROOT"), drive_module=drive,
    mountpoint=os.environ.get("COT_DRIVE_MOUNTPOINT", "/content/drive"))
STUDY = sessao.Session(REPO, STATE, CODE_COMMIT, CODE_HASHES, on_colab=ON_COLAB, technical_test=TECHNICAL_TEST)
Blocked = sessao.StageBlocked
print("Código:", CODE_COMMIT, "| Estudo:", STUDY_ID, "| Teste técnico:", TECHNICAL_TEST)


Armazenamento persistente verificado: /content/drive/MyDrive/CoT_Obfuscation_Lab/prospectivo_v4/estudo-prospectivo-2026-v4 (livre: 213.46 GiB)


Código: 9066d811e656cb7c0d3c2120ba06c7ae356691da | Estudo: estudo-prospectivo-2026-v4 | Teste técnico: False


In [2]:
status = STUDY.execute("status")
if not status["all_training_complete"]:
    print("ETAPA BLOQUEADA: TEST só é aberto após todos os treinos:", status["training"], "de", status["expected"])
else:
    try:
        for mode in status["expected"]:
            progress = STUDY.execute("evaluate", mode=mode, max_jobs=MAX_JOBS_PER_SESSION)
            print(f"{mode.upper()}: {progress['completed']}/{progress['total']} avaliações ({progress['status']}).")
    except Blocked as reason:
        print("ETAPA BLOQUEADA:", reason)
STUDY.export(download=False)
STUDY.status()


Etapa "status" concluída; resultado completo em /content/drive/MyDrive/CoT_Obfuscation_Lab/prospectivo_v4/estudo-prospectivo-2026-v4/operations/status_sft.json.


GPU: NVIDIA A100-SXM4-80GB | 79.3 GiB | PyTorch 2.11.0+cu130 | CUDA 13.0


Etapa "evaluate" concluída; resultado completo em /content/drive/MyDrive/CoT_Obfuscation_Lab/prospectivo_v4/estudo-prospectivo-2026-v4/operations/evaluate_sft.json.


SFT: 24/120 avaliações (resume_next_session).


ZIP salvo: /content/drive/MyDrive/CoT_Obfuscation_Lab/prospectivo_v4/estudo-prospectivo-2026-v4/downloads/resultados_v4.zip (858 KiB, SHA256 cd3dfa58fe64cd0e…)


Etapa "status" concluída; resultado completo em /content/drive/MyDrive/CoT_Obfuscation_Lab/prospectivo_v4/estudo-prospectivo-2026-v4/operations/status_sft.json.



SITUAÇÃO DO ESTUDO
  00 Protocolo e dados         CONCLUÍDA        
  Triagem DEV (65% por nível)  CONCLUÍDA        sft/A=97.5% sft/C=68.3% grpo/low=95.0% grpo/high=90.8%
  Registro público             CONCLUÍDA        declared_external_timestamp_requires_audit
  01 Treinos SFT               CONCLUÍDA        120/120
  02 Treinos GRPO              EXCLUÍDA         braço excluído antes do registro (ver protocolo)
  03 Avaliação TEST            EM ANDAMENTO     24/120
  Armazenamento: /content/drive/MyDrive/CoT_Obfuscation_Lab/prospectivo_v4/estudo-prospectivo-2026-v4 | persistente: True


[('00 Protocolo e dados', 'CONCLUÍDA', ''),
 ('Triagem DEV (65% por nível)',
  'CONCLUÍDA',
  'sft/A=97.5% sft/C=68.3% grpo/low=95.0% grpo/high=90.8%'),
 ('Registro público',
  'CONCLUÍDA',
  'declared_external_timestamp_requires_audit'),
 ('01 Treinos SFT', 'CONCLUÍDA', '120/120'),
 ('02 Treinos GRPO',
  'EXCLUÍDA',
  'braço excluído antes do registro (ver protocolo)'),
 ('03 Avaliação TEST', 'EM ANDAMENTO', '24/120')]

**Situação das etapas.** CONCLUÍDA: arquivos verificados por hash. BLOQUEADA: um critério do
protocolo ou um recurso impede a etapa (o motivo é impresso). NÃO EXECUTADA / PENDENTE: ainda
não iniciada. Reexecutar retoma o mesmo `STUDY_ID`; trocar o ID inicia outro estudo.
Cópias antigas salvas no Colab mantêm o commit antigo: abra sempre pelo link do README.
